# 📚 Guía de IA Generativa - RAG (Retrieval Augmented Generation)



En este notebook aprenderás:
- ✅ Qué es RAG y por qué resuelve las limitaciones de los LLMs
- ✅ Técnicas de chunking (división de documentos)
- ✅ Cómo crear embeddings para búsqueda semántica
- ✅ Estrategias de retrieval (recuperación de información)
- ✅ Cómo componer el prompt final con contexto
- ✅ Construir un sistema RAG completo paso a paso

**⚠️ IMPORTANTE - SEGURIDAD:**
- ❌ NUNCA compartas tus API keys
- ✅ Usa variables de entorno o widgets seguros

---
**📋 Requisitos previos:**
- Haber completado los Notebooks 1, 2 y 3
- Entender qué son los embeddings (Notebook 2)
- API key de Gemini (GRATIS) para la generación

# 0. SETUP Y CONTEXTO

## 📚 Instalar Librerías

In [ ]:
# 📦 Instalación de dependencias
import sys
import subprocess
print("🔧 Instalando paquetes necesarios...")
print("=" * 60)

packages = [
    ("google-genai", "🌟 Cliente de Google para Gemini (GRATIS)"),
    ("sentence-transformers", "🤗 Embeddings locales gratuitos"),
    ("numpy", "🔢 Operaciones numéricas"),
    ("scikit-learn", "📊 Similitud coseno y métricas"),
    ("ipywidgets", "🎨 Widgets interactivos"),
    ("tiktoken", "🎫 Contador de tokens"),
]

for package, descripcion in packages:
    try:
        print(f"📥 Instalando {package}...")
        print(f"   ℹ️  {descripcion}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} instalado correctamente")
    except Exception as e:
        print(f"⚠️ Error instalando {package}: {e}")

print("=" * 60)
print("✨ ¡Instalación completada!")

## 📚 Importar Librerías

In [ ]:
# Importaciones estándar
import os
import sys
import re
import json
from datetime import datetime
from typing import List, Dict, Tuple, Optional

# Numérico
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Visualización
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from ipywidgets import Layout, Button, VBox, HBox, Textarea, Password, Output

# Tokenización
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base")
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False

# Embeddings locales
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("⚠️ sentence-transformers no disponible")

# Google Gemini
try:
    from google import genai
    from google.genai import types
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print("⚠️ Google Gemini no disponible")

# SSL
VERIFICAR_SSL = False
if not VERIFICAR_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("✅ Librerías importadas correctamente")
print(f"📊 Python: {sys.version.split()[0]}")
print(f"🤗 Sentence Transformers: {'✅' if SENTENCE_TRANSFORMERS_AVAILABLE else '❌'}")

## 🔐 Configuración y Modelos

In [ ]:
# 🤗 Cargar modelo de embeddings local (GRATIS)
print("🔄 Cargando modelo de embeddings local...")
print("   (Primera vez descarga ~470MB, luego es instantáneo)")

if SENTENCE_TRANSFORMERS_AVAILABLE:
    modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print(f"✅ Modelo cargado: paraphrase-multilingual-MiniLM-L12-v2")
    print(f"📐 Dimensión de embeddings: {modelo_embeddings.get_sentence_embedding_dimension()}")
else:
    print("❌ No se pudo cargar el modelo de embeddings")

In [ ]:
# 🔐 Widget para API Key de Gemini (para la generación)
print("🔑 Configuración de API Key de Gemini")
print("=" * 60)
print("Necesaria para la parte de GENERACIÓN del RAG.")
print("📎 Obtén tu key GRATIS en: https://makersuite.google.com/app/apikey")
print()

gemini_key_widget = Password(
    placeholder='AIza...',
    description='Gemini:',
    layout=Layout(width='500px')
)

save_button = Button(
    description='💾 Guardar y Conectar',
    button_style='success',
    layout=Layout(width='200px')
)

status_output = Output()
gemini_client = None

def save_and_connect(b):
    global gemini_client
    with status_output:
        status_output.clear_output()
        if gemini_key_widget.value and gemini_key_widget.value.startswith('AIza'):
            os.environ['GEMINI_API_KEY'] = gemini_key_widget.value
            try:
                gemini_client = genai.Client(api_key=gemini_key_widget.value)
                response = gemini_client.models.generate_content(
                    model='gemini-2.0-flash-lite',
                    contents='Di solo "OK"'
                )
                print("✅ ¡Conexión exitosa con Gemini!")
            except Exception as e:
                print(f"❌ Error: {e}")
                gemini_client = None
        else:
            print("⚠️ Key inválida")

save_button.on_click(save_and_connect)
display(VBox([gemini_key_widget, save_button, status_output]))

In [ ]:
# 🛠️ Función auxiliar para llamar a Gemini
def llamar_gemini(prompt: str, max_tokens: int = 1000) -> str:
    """Llama a Gemini y devuelve la respuesta."""
    if gemini_client is None:
        return "❌ ERROR: Configura tu API key de Gemini primero"
    
    try:
        response = gemini_client.models.generate_content(
            model='gemini-2.0-flash-lite',
            contents=prompt,
            config=types.GenerateContentConfig(
                max_output_tokens=max_tokens,
                temperature=0.3
            )
        )
        return response.text
    except Exception as e:
        return f"❌ ERROR: {str(e)}"

print("✅ Funciones auxiliares cargadas")

---
# 🎓 1. ¿QUÉ ES RAG Y POR QUÉ LO NECESITAMOS?

## 1.1 El problema de los LLMs

Los LLMs tienen **limitaciones importantes**:

```
┌─────────────────────────────────────────────────────────────┐
│                 LIMITACIONES DE LOS LLMs                   │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  📅 CONOCIMIENTO DESACTUALIZADO                             │
│     El modelo solo sabe lo que había hasta su fecha         │
│     de entrenamiento. No conoce eventos recientes.          │
│                                                             │
│  🏢 NO CONOCE TUS DATOS PRIVADOS                            │
│     No sabe nada sobre tu empresa, tus documentos,          │
│     tus bases de datos internas.                            │
│                                                             │
│  🎭 ALUCINACIONES                                           │
│     Cuando no sabe algo, puede inventar respuestas          │
│     que parecen correctas pero son falsas.                  │
│                                                             │
│  📏 LÍMITE DE CONTEXTO                                      │
│     No puedes pasarle millones de documentos                │
│     en cada consulta.                                       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 1.2 La solución: RAG

**RAG = Retrieval Augmented Generation** (Generación Aumentada por Recuperación)

La idea es simple:
1. **BUSCA** información relevante en tus documentos
2. **AÑADE** esa información al prompt
3. **GENERA** la respuesta basándose en el contexto encontrado

```
┌─────────────────────────────────────────────────────────────┐
│                    FLUJO DE RAG                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   👤 Usuario: "¿Cuál es la política de vacaciones?"         │
│                           │                                 │
│                           ▼                                 │
│   ┌─────────────────────────────────────┐                   │
│   │  1️⃣ RETRIEVAL (Búsqueda)            │                   │
│   │  Busca en documentos de la empresa  │                   │
│   └─────────────────────────────────────┘                   │
│                           │                                 │
│                           ▼                                 │
│   📄 Fragmentos encontrados:                                │
│   "Los empleados tienen 23 días de vacaciones..."           │
│                           │                                 │
│                           ▼                                 │
│   ┌─────────────────────────────────────┐                   │
│   │  2️⃣ AUGMENTATION (Aumentar)         │                   │
│   │  Añade contexto al prompt           │                   │
│   └─────────────────────────────────────┘                   │
│                           │                                 │
│                           ▼                                 │
│   ┌─────────────────────────────────────┐                   │
│   │  3️⃣ GENERATION (Generar)            │                   │
│   │  LLM responde con el contexto       │                   │
│   └─────────────────────────────────────┘                   │
│                           │                                 │
│                           ▼                                 │
│   🤖 "Según la política de la empresa, tienes              │
│       23 días de vacaciones al año..."                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 1.3 Componentes de un sistema RAG

| Componente | Función | Técnicas |
|------------|---------|----------|
| **1. Chunking** | Dividir documentos en trozos | Por tamaño, párrafos, semántica |
| **2. Embedding** | Convertir texto a vectores | OpenAI, HuggingFace, Cohere |
| **3. Indexación** | Almacenar vectores | FAISS, Pinecone, ChromaDB |
| **4. Retrieval** | Buscar chunks relevantes | Similitud coseno, BM25, híbrido |
| **5. Generación** | Crear respuesta final | GPT, Gemini, Claude |

---
# ✂️ 2. CHUNKING: Dividir documentos

## 2.1 ¿Por qué necesitamos chunking?


Los documentos largos no caben en el contexto del LLM, y aunque cupieran:
- **Más tokens = más coste** 💰
- **Información irrelevante = peores respuestas**
- **Los embeddings funcionan mejor con textos cortos**

 📏 Tamaño ideal de chunk:
- **Muy pequeño** (< 100 tokens): Pierde contexto
- **Muy grande** (> 1000 tokens): Embeddings menos precisos
- **Sweet spot**: 200-500 tokens

In [ ]:
# 📄 Documento de ejemplo para practicar

DOCUMENTO_EJEMPLO = """
# Manual de Empleado - TechCorp 2026

## Capítulo 1: Horarios y Asistencia

El horario laboral estándar es de 9:00 a 18:00, con una hora de comida. 
Los empleados pueden solicitar horario flexible mediante el portal de RRHH.
El teletrabajo está permitido hasta 3 días por semana previa aprobación del manager.

Las ausencias deben comunicarse antes de las 9:00 al supervisor directo.
En caso de enfermedad, se requiere justificante médico a partir del tercer día.

## Capítulo 2: Vacaciones y Permisos

Cada empleado tiene derecho a 23 días laborables de vacaciones al año.
Las vacaciones deben solicitarse con al menos 2 semanas de antelación.
El período de Navidad (24 dic - 1 ene) tiene restricciones especiales.

Permisos retribuidos:
- Matrimonio: 15 días naturales
- Nacimiento de hijo: 16 semanas (ampliable)
- Fallecimiento familiar directo: 3-5 días según distancia
- Mudanza: 1 día

## Capítulo 3: Beneficios y Compensación

El salario se abona el último día hábil de cada mes.
Pagas extras: junio y diciembre (prorrateadas).

Beneficios incluidos:
- Seguro médico privado (empleado + familia)
- Ticket restaurante: 11€/día laborable
- Formación: hasta 2.000€/año en cursos aprobados
- Gimnasio: 50% descuento en cadenas asociadas

## Capítulo 4: Código de Conducta

Se espera que todos los empleados mantengan un comportamiento profesional.
El acoso de cualquier tipo está estrictamente prohibido.
La información confidencial de la empresa no debe compartirse externamente.

El uso de equipos de la empresa para fines personales debe ser moderado.
Las redes sociales corporativas deben usarse de forma profesional.

## Capítulo 5: Desarrollo Profesional

Evaluaciones de desempeño: semestrales (junio y diciembre).
Los objetivos se definen en Q1 junto con el manager.
Las promociones se revisan anualmente en el comité de talento.

Programa de mentoring disponible para todos los empleados.
Rotación entre departamentos posible tras 2 años en el puesto.
"""

print("📄 DOCUMENTO DE EJEMPLO CARGADO")
print(f"   Caracteres: {len(DOCUMENTO_EJEMPLO):,}")
print(f"   Palabras: {len(DOCUMENTO_EJEMPLO.split()):,}")
if TIKTOKEN_AVAILABLE:
    print(f"   Tokens: {len(tokenizer.encode(DOCUMENTO_EJEMPLO)):,}")

## 2.2 Método 1: Chunking por tamaño fijo

In [ ]:
# ✂️ CHUNKING POR TAMAÑO FIJO

def chunk_por_tamaño(texto: str, tamaño_chunk: int = 500, overlap: int = 50) -> List[str]:
    """
    Divide el texto en chunks de tamaño fijo con overlap.
    
    Args:
        texto: Texto a dividir
        tamaño_chunk: Número de caracteres por chunk
        overlap: Caracteres de solapamiento entre chunks
    
    Returns:
        Lista de chunks
    """
    chunks = []
    inicio = 0
    
    while inicio < len(texto):
        fin = inicio + tamaño_chunk
        chunk = texto[inicio:fin]
        chunks.append(chunk.strip())
        inicio = fin - overlap  # Retrocedemos para el overlap
    
    return chunks

# Probar
chunks_fijos = chunk_por_tamaño(DOCUMENTO_EJEMPLO, tamaño_chunk=500, overlap=50)

print("✂️ CHUNKING POR TAMAÑO FIJO")
print(f"   Tamaño: 500 caracteres | Overlap: 50")
print(f"   Chunks generados: {len(chunks_fijos)}")
print()

for i, chunk in enumerate(chunks_fijos[:3], 1):
    print(f"📦 Chunk {i} ({len(chunk)} chars):")
    print(f"   '{chunk[:100]}...'")
    print()

## 2.3 Método 2: Chunking por párrafos/secciones

In [ ]:
# ✂️ CHUNKING POR SECCIONES (HEADERS)

def chunk_por_secciones(texto: str, separador: str = "## ") -> List[Dict]:
    """
    Divide el texto por secciones marcadas con headers.
    Preserva metadatos (título de sección).
    
    Returns:
        Lista de diccionarios con 'titulo' y 'contenido'
    """
    chunks = []
    secciones = texto.split(separador)
    
    for seccion in secciones:
        seccion = seccion.strip()
        if not seccion:
            continue
        
        # Extraer título (primera línea)
        lineas = seccion.split('\n')
        titulo = lineas[0].strip()
        contenido = '\n'.join(lineas[1:]).strip()
        
        if contenido:  # Solo si hay contenido
            chunks.append({
                'titulo': titulo,
                'contenido': contenido
            })
    
    return chunks

# Probar
chunks_secciones = chunk_por_secciones(DOCUMENTO_EJEMPLO)

print("✂️ CHUNKING POR SECCIONES")
print(f"   Separador: '## '")
print(f"   Secciones encontradas: {len(chunks_secciones)}")
print()

for i, chunk in enumerate(chunks_secciones, 1):
    print(f"📦 Sección {i}: {chunk['titulo']}")
    print(f"   Contenido: {len(chunk['contenido'])} chars")
    print(f"   Preview: '{chunk['contenido'][:80]}...'")
    print()

## 2.4 Método 3: Chunking por oraciones

In [ ]:
# ✂️ CHUNKING POR ORACIONES (con agrupación)

def chunk_por_oraciones(texto: str, oraciones_por_chunk: int = 3, overlap_oraciones: int = 1) -> List[str]:
    """
    Divide el texto en oraciones y las agrupa.
    
    Args:
        texto: Texto a dividir
        oraciones_por_chunk: Número de oraciones por chunk
        overlap_oraciones: Oraciones de solapamiento
    """
    # Dividir por puntos (simplificado)
    oraciones = re.split(r'(?<=[.!?])\s+', texto)
    oraciones = [o.strip() for o in oraciones if o.strip()]
    
    chunks = []
    i = 0
    
    while i < len(oraciones):
        chunk_oraciones = oraciones[i:i + oraciones_por_chunk]
        chunk = ' '.join(chunk_oraciones)
        if chunk:
            chunks.append(chunk)
        i += oraciones_por_chunk - overlap_oraciones
    
    return chunks

# Probar
chunks_oraciones = chunk_por_oraciones(DOCUMENTO_EJEMPLO, oraciones_por_chunk=4, overlap_oraciones=1)

print("✂️ CHUNKING POR ORACIONES")
print(f"   Oraciones por chunk: 4 | Overlap: 1")
print(f"   Chunks generados: {len(chunks_oraciones)}")
print()

for i, chunk in enumerate(chunks_oraciones[:3], 1):
    print(f"📦 Chunk {i}:")
    print(f"   '{chunk[:150]}...'")
    print()

## 2.5 Método 4: Chunking recursivo (más sofisticado)

In [ ]:
# ✂️ CHUNKING RECURSIVO (estilo LangChain)

def chunk_recursivo(texto: str, max_chars: int = 500, separadores: List[str] = None) -> List[str]:
    """
    Divide recursivamente usando múltiples separadores.
    Intenta mantener unidades semánticas (párrafos > oraciones > palabras).
    
    Args:
        texto: Texto a dividir
        max_chars: Máximo de caracteres por chunk
        separadores: Lista de separadores ordenados por preferencia
    """
    if separadores is None:
        separadores = ["\n\n", "\n", ". ", " "]
    
    # Si el texto es suficientemente pequeño, devolverlo
    if len(texto) <= max_chars:
        return [texto.strip()] if texto.strip() else []
    
    # Probar cada separador
    for sep in separadores:
        if sep in texto:
            partes = texto.split(sep)
            chunks = []
            chunk_actual = ""
            
            for parte in partes:
                # Si añadir esta parte excede el límite
                if len(chunk_actual) + len(parte) + len(sep) > max_chars:
                    if chunk_actual:
                        chunks.append(chunk_actual.strip())
                    # Si la parte sola es muy grande, dividirla recursivamente
                    if len(parte) > max_chars:
                        chunks.extend(chunk_recursivo(parte, max_chars, separadores[separadores.index(sep)+1:]))
                        chunk_actual = ""
                    else:
                        chunk_actual = parte
                else:
                    chunk_actual = chunk_actual + sep + parte if chunk_actual else parte
            
            if chunk_actual.strip():
                chunks.append(chunk_actual.strip())
            
            return chunks
    
    # Si ningún separador funciona, cortar por caracteres
    return [texto[i:i+max_chars].strip() for i in range(0, len(texto), max_chars)]

# Probar
chunks_recursivo = chunk_recursivo(DOCUMENTO_EJEMPLO, max_chars=400)

print("✂️ CHUNKING RECURSIVO")
print(f"   Máximo: 400 chars")
print(f"   Separadores: [párrafo, línea, oración, espacio]")
print(f"   Chunks generados: {len(chunks_recursivo)}")
print()

for i, chunk in enumerate(chunks_recursivo[:4], 1):
    print(f"📦 Chunk {i} ({len(chunk)} chars):")
    print(f"   '{chunk[:100]}...'")
    print()

## 2.6 Comparativa de métodos

In [ ]:
# 📊 COMPARATIVA DE MÉTODOS DE CHUNKING

print("📊 COMPARATIVA DE MÉTODOS DE CHUNKING")
print("=" * 60)
print()

metodos = [
    ("Tamaño fijo (500 chars)", chunks_fijos),
    ("Por secciones", [c['contenido'] for c in chunks_secciones]),
    ("Por oraciones (4)", chunks_oraciones),
    ("Recursivo (400 chars)", chunks_recursivo)
]

print(f"{'Método':<25} {'Chunks':>8} {'Avg chars':>12} {'Min':>8} {'Max':>8}")
print("-" * 65)

for nombre, chunks in metodos:
    if chunks:
        tamaños = [len(c) for c in chunks]
        print(f"{nombre:<25} {len(chunks):>8} {np.mean(tamaños):>12.1f} {min(tamaños):>8} {max(tamaños):>8}")

print()
print("💡 RECOMENDACIONES:")
print("   - Documentos estructurados (manuales): Chunking por secciones")
print("   - Texto libre (artículos): Chunking recursivo o por oraciones")
print("   - Código fuente: Chunking por funciones/clases")
print("   - Siempre usa overlap para no perder contexto en los bordes")

---
# 🔢 3. EMBEDDINGS PARA RAG

## 3.1 Generar embeddings de los chunks

In [ ]:
# 🔢 CREAR EMBEDDINGS PARA NUESTROS CHUNKS

# Usamos el chunking por secciones para este ejemplo
chunks_para_rag = []

for chunk in chunks_secciones:
    # Combinar título y contenido para el embedding
    texto_completo = f"{chunk['titulo']}\n{chunk['contenido']}"
    chunks_para_rag.append({
        'id': len(chunks_para_rag),
        'titulo': chunk['titulo'],
        'contenido': chunk['contenido'],
        'texto_para_embedding': texto_completo
    })

print("🔢 GENERANDO EMBEDDINGS")
print("=" * 50)

# Generar embeddings
textos = [c['texto_para_embedding'] for c in chunks_para_rag]
embeddings = modelo_embeddings.encode(textos)

# Guardar en los chunks
for i, chunk in enumerate(chunks_para_rag):
    chunk['embedding'] = embeddings[i]

print(f"✅ Embeddings generados: {len(chunks_para_rag)}")
print(f"📐 Dimensión: {embeddings.shape[1]}")
print()

for chunk in chunks_para_rag:
    print(f"   📦 Chunk {chunk['id']}: {chunk['titulo']}")
    print(f"      Embedding: [{chunk['embedding'][:3].round(3)}...]")

## 3.2 Nuestra "base de datos" vectorial simple

In [ ]:
# 🗄️ BASE DE DATOS VECTORIAL SIMPLE

class VectorStoreSimple:
    """
    Base de datos vectorial minimalista para entender el concepto.
    En producción usarías FAISS, Pinecone, ChromaDB, etc.
    """
    
    def __init__(self, modelo_embeddings):
        self.modelo = modelo_embeddings
        self.documentos = []  # Lista de {id, texto, metadata, embedding}
    
    def agregar(self, textos: List[str], metadatas: List[dict] = None):
        """Agrega documentos a la base de datos."""
        embeddings = self.modelo.encode(textos)
        
        for i, (texto, emb) in enumerate(zip(textos, embeddings)):
            self.documentos.append({
                'id': len(self.documentos),
                'texto': texto,
                'metadata': metadatas[i] if metadatas else {},
                'embedding': emb
            })
    
    def buscar(self, consulta: str, top_k: int = 3) -> List[Tuple[dict, float]]:
        """
        Busca los documentos más similares a la consulta.
        
        Returns:
            Lista de (documento, similitud)
        """
        # Embedding de la consulta
        emb_consulta = self.modelo.encode(consulta)
        
        # Calcular similitudes
        similitudes = []
        for doc in self.documentos:
            sim = cosine_similarity([emb_consulta], [doc['embedding']])[0][0]
            similitudes.append((doc, sim))
        
        # Ordenar por similitud descendente
        similitudes.sort(key=lambda x: x[1], reverse=True)
        
        return similitudes[:top_k]

# Crear nuestra base de datos
vector_store = VectorStoreSimple(modelo_embeddings)

# Agregar los chunks
textos = [c['texto_para_embedding'] for c in chunks_para_rag]
metadatas = [{'titulo': c['titulo']} for c in chunks_para_rag]
vector_store.agregar(textos, metadatas)

print("🗄️ BASE DE DATOS VECTORIAL CREADA")
print(f"   Documentos indexados: {len(vector_store.documentos)}")

---
# 🔍 4. RETRIEVAL: Buscar información relevante

## 4.1 Búsqueda semántica básica

In [ ]:
# 🔍 BÚSQUEDA SEMÁNTICA

def buscar_relevante(consulta: str, top_k: int = 3):
    """Busca y muestra los chunks más relevantes."""
    print(f"🔍 BÚSQUEDA: '{consulta}'")
    print("=" * 60)
    
    resultados = vector_store.buscar(consulta, top_k=top_k)
    
    for i, (doc, similitud) in enumerate(resultados, 1):
        barra = "█" * int(similitud * 20)
        print(f"\n#{i} Similitud: {similitud:.3f} {barra}")
        print(f"   📁 Sección: {doc['metadata'].get('titulo', 'N/A')}")
        print(f"   📝 Contenido: {doc['texto'][:200]}...")
    
    return resultados

# Probar búsquedas
buscar_relevante("¿Cuántos días de vacaciones tengo?")

In [ ]:
# Más búsquedas de prueba
buscar_relevante("teletrabajo desde casa")

In [ ]:
buscar_relevante("beneficios para empleados")

## 4.2 Estrategias de retrieval avanzadas

In [ ]:
# 🔍 RETRIEVAL CON UMBRAL DE SIMILITUD

def buscar_con_umbral(consulta: str, umbral: float = 0.3, max_resultados: int = 5) -> List[dict]:
    """
    Busca documentos que superen un umbral de similitud.
    Evita devolver resultados irrelevantes.
    """
    resultados = vector_store.buscar(consulta, top_k=max_resultados)
    
    # Filtrar por umbral
    relevantes = [(doc, sim) for doc, sim in resultados if sim >= umbral]
    
    print(f"🔍 Búsqueda: '{consulta}'")
    print(f"   Resultados totales: {len(resultados)}")
    print(f"   Sobre umbral ({umbral}): {len(relevantes)}")
    
    return relevantes

# Probar
buscar_con_umbral("horario de trabajo", umbral=0.4)

In [ ]:
# 🔍 RETRIEVAL CON RE-RANKING

def buscar_con_reranking(consulta: str, top_k: int = 3) -> List[dict]:
    """
    Búsqueda en dos fases:
    1. Retrieval inicial (más candidatos)
    2. Re-ranking con criterios adicionales
    """
    # Fase 1: Retrieval amplio
    candidatos = vector_store.buscar(consulta, top_k=10)
    
    # Fase 2: Re-ranking (aquí podrías usar otro modelo más preciso)
    # Por simplicidad, usamos la longitud del match de palabras clave
    palabras_consulta = set(consulta.lower().split())
    
    reranked = []
    for doc, sim_embedding in candidatos:
        palabras_doc = set(doc['texto'].lower().split())
        overlap = len(palabras_consulta & palabras_doc)
        
        # Combinar similitud semántica con overlap de palabras
        score_final = sim_embedding * 0.7 + (overlap / len(palabras_consulta)) * 0.3
        reranked.append((doc, score_final, sim_embedding))
    
    # Ordenar por score final
    reranked.sort(key=lambda x: x[1], reverse=True)
    
    print(f"🔍 Búsqueda con re-ranking: '{consulta}'")
    print("=" * 60)
    
    for i, (doc, score, sim_orig) in enumerate(reranked[:top_k], 1):
        print(f"\n#{i} Score final: {score:.3f} (embedding: {sim_orig:.3f})")
        print(f"   📁 {doc['metadata'].get('titulo', 'N/A')}")
    
    return reranked[:top_k]

buscar_con_reranking("días de vacaciones")

---
# 🤖 5. GENERACIÓN: Componer el prompt final

## 5.1 Plantilla de prompt RAG

In [ ]:
# 📝 PLANTILLA DE PROMPT RAG

PROMPT_RAG_TEMPLATE = """Eres un asistente de RRHH que responde preguntas basándose ÚNICAMENTE 
en la información proporcionada. Si la información no está en el contexto, di que no lo sabes.

CONTEXTO (información del manual de empleado):
---
{contexto}
---

PREGUNTA: {pregunta}

INSTRUCCIONES:
- Responde SOLO basándote en el contexto proporcionado
- Si la información no está en el contexto, di "No tengo información sobre esto en el manual"
- Sé conciso pero completo
- Cita la sección relevante si es posible

RESPUESTA:"""

def generar_respuesta_rag(pregunta: str, contextos: List[Tuple[dict, float]]) -> str:
    """
    Genera una respuesta usando el contexto recuperado.
    """
    # Construir el contexto
    contexto_texto = "\n\n".join([
        f"[{doc['metadata'].get('titulo', 'Sección')}]\n{doc['texto'][:500]}"
        for doc, _ in contextos
    ])
    
    # Construir el prompt
    prompt = PROMPT_RAG_TEMPLATE.format(
        contexto=contexto_texto,
        pregunta=pregunta
    )
    
    # Generar respuesta
    respuesta = llamar_gemini(prompt)
    
    return respuesta, prompt

print("✅ Plantilla RAG configurada")

## 5.2 Pipeline RAG completo

In [ ]:
# 🔄 PIPELINE RAG COMPLETO

def rag_pipeline(pregunta: str, top_k: int = 2, mostrar_detalles: bool = True, mostrar_prompt_completo: bool = True) -> str:
    """
    Pipeline RAG completo: Retrieval → Augmentation → Generation
    """
    print("🔄 PIPELINE RAG")
    print("=" * 60)
    print(f"❓ PREGUNTA: {pregunta}")
    print()
    
    # 1️⃣ RETRIEVAL
    print("1️⃣ RETRIEVAL - Buscando información relevante...")
    contextos = vector_store.buscar(pregunta, top_k=top_k)
    
    if mostrar_detalles:
        for i, (doc, sim) in enumerate(contextos, 1):
            print(f"   📄 Chunk {i}: {doc['metadata'].get('titulo', 'N/A')} (sim: {sim:.3f})")
    print()
    
    # 2️⃣ AUGMENTATION
    print("2️⃣ AUGMENTATION - Construyendo prompt con contexto...")

    contexto_texto = "\n\n".join([
        f"[{doc['metadata'].get('titulo', 'Sección')}]\n{doc['texto']}"
        for doc, _ in contextos
    ])

    prompt_usado = PROMPT_RAG_TEMPLATE.format(
        contexto=contexto_texto,
        pregunta=pregunta
    )

    if mostrar_detalles:
        tokens_prompt = len(tokenizer.encode(prompt_usado)) if TIKTOKEN_AVAILABLE else "N/A"
        print(f"   📊 Tokens del prompt: {tokens_prompt}")

    if mostrar_prompt_completo:
        print()
        print("🧾 PROMPT COMPLETO (con contexto):")
        print("-" * 60)
        print(prompt_usado)
        print("-" * 60)
    print()
    
    # 3️⃣ GENERATION
    print("3️⃣ GENERATION - Generando respuesta...")
    respuesta = llamar_gemini(prompt_usado)
    print()
    print("📤 RESPUESTA:")
    print("-" * 40)
    print(respuesta)
    print("-" * 40)
    
    return respuesta

# Probar el pipeline
rag_pipeline("¿Cuántos días de vacaciones tengo al año?")

In [ ]:
# Más ejemplos del pipeline
rag_pipeline("¿Puedo trabajar desde casa? ¿Cuántos días?")

In [ ]:
rag_pipeline("¿Qué beneficios de salud tengo?")

In [ ]:
# Pregunta que NO está en el documento
rag_pipeline("¿Cuál es el código de vestimenta?")

## 5.3 Widget interactivo de RAG

In [ ]:
# 🎮 WIDGET INTERACTIVO DE RAG

pregunta_input = Textarea(
    value='',
    placeholder='Escribe tu pregunta sobre el manual de empleado...',
    description='Pregunta:',
    layout=Layout(width='600px', height='60px')
)

preguntar_btn = Button(
    description='🔍 Preguntar al RAG',
    button_style='primary',
    layout=Layout(width='200px')
)

output_rag = Output()

def ejecutar_rag(b):
    with output_rag:
        output_rag.clear_output()
        pregunta = pregunta_input.value.strip()
        if pregunta:
            rag_pipeline(pregunta)
        else:
            print("⚠️ Escribe una pregunta")

preguntar_btn.on_click(ejecutar_rag)

print("🎮 ASISTENTE RAG INTERACTIVO")
print("Pregunta cualquier cosa sobre el manual de empleado de TechCorp.")
print()
print("💡 Ejemplos de preguntas:")
print("   - ¿Cuántos días de permiso por boda?")
print("   - ¿Cómo pido vacaciones?")
print("   - ¿Qué pasa si llego tarde?")
print("   - ¿Hay ayuda para formación?")
print()

display(VBox([pregunta_input, preguntar_btn, output_rag]))

---
# 🏗️ 6. RAG AVANZADO: Mejoras y optimizaciones

## 6.1 Añadir metadatos para filtrado

In [ ]:
# 🏷️ RAG CON METADATOS

class VectorStoreConMetadatos:
    """Vector store que permite filtrar por metadatos."""
    
    def __init__(self, modelo_embeddings):
        self.modelo = modelo_embeddings
        self.documentos = []
    
    def agregar(self, texto: str, metadata: dict):
        """Agrega un documento con metadatos."""
        embedding = self.modelo.encode(texto)
        self.documentos.append({
            'id': len(self.documentos),
            'texto': texto,
            'metadata': metadata,
            'embedding': embedding
        })
    
    def buscar(self, consulta: str, filtros: dict = None, top_k: int = 3):
        """
        Busca con filtros opcionales por metadatos.
        
        Args:
            consulta: Texto de búsqueda
            filtros: Dict de {campo: valor} para filtrar
            top_k: Número de resultados
        """
        emb_consulta = self.modelo.encode(consulta)
        
        # Filtrar documentos
        docs_filtrados = self.documentos
        if filtros:
            for campo, valor in filtros.items():
                docs_filtrados = [
                    d for d in docs_filtrados 
                    if d['metadata'].get(campo) == valor
                ]
        
        # Calcular similitudes
        resultados = []
        for doc in docs_filtrados:
            sim = cosine_similarity([emb_consulta], [doc['embedding']])[0][0]
            resultados.append((doc, sim))
        
        resultados.sort(key=lambda x: x[1], reverse=True)
        return resultados[:top_k]

# Crear store con metadatos
store_metadatos = VectorStoreConMetadatos(modelo_embeddings)

# Agregar documentos con categorías
for chunk in chunks_secciones:
    # Inferir categoría del título
    titulo = chunk['titulo'].lower()
    if 'vacaciones' in titulo or 'permiso' in titulo:
        categoria = 'tiempo_libre'
    elif 'horario' in titulo or 'asistencia' in titulo:
        categoria = 'horarios'
    elif 'beneficio' in titulo or 'compensación' in titulo:
        categoria = 'beneficios'
    elif 'conducta' in titulo:
        categoria = 'normativa'
    else:
        categoria = 'general'
    
    store_metadatos.agregar(
        texto=f"{chunk['titulo']}\n{chunk['contenido']}",
        metadata={'titulo': chunk['titulo'], 'categoria': categoria}
    )

print("🏷️ VECTOR STORE CON METADATOS CREADO")
print()

# Búsqueda filtrada
print("🔍 Búsqueda filtrada por categoría 'tiempo_libre':")
resultados = store_metadatos.buscar("días libres", filtros={'categoria': 'tiempo_libre'})
for doc, sim in resultados:
    print(f"   📄 {doc['metadata']['titulo']} (sim: {sim:.3f})")

## 6.2 Conversación con historial

In [ ]:
# 💬 RAG CON HISTORIAL DE CONVERSACIÓN

class RAGConHistorial:
    """Sistema RAG que mantiene contexto de la conversación."""
    
    def __init__(self, vector_store, max_historial: int = 5):
        self.store = vector_store
        self.historial = []  # Lista de (pregunta, respuesta)
        self.max_historial = max_historial
    
    def _construir_contexto_historial(self) -> str:
        """Construye un resumen del historial reciente."""
        if not self.historial:
            return ""
        
        lineas = ["CONVERSACIÓN PREVIA:"]
        for preg, resp in self.historial[-3:]:  # Últimas 3
            lineas.append(f"Usuario: {preg}")
            lineas.append(f"Asistente: {resp[:100]}...")
        
        return "\n".join(lineas)
    
    def preguntar(self, pregunta: str) -> str:
        """Procesa una pregunta considerando el historial."""
        # Retrieval
        contextos = self.store.buscar(pregunta, top_k=2)
        
        contexto_docs = "\n\n".join([
            f"[{doc['metadata'].get('titulo', 'Doc')}]\n{doc['texto'][:400]}"
            for doc, _ in contextos
        ])
        
        historial_texto = self._construir_contexto_historial()
        
        # Prompt con historial
        prompt = f"""Eres un asistente de RRHH.

{historial_texto}

CONTEXTO (del manual):
{contexto_docs}

PREGUNTA ACTUAL: {pregunta}

Responde considerando la conversación previa si es relevante."""
        
        respuesta = llamar_gemini(prompt)
        
        # Guardar en historial
        self.historial.append((pregunta, respuesta))
        if len(self.historial) > self.max_historial:
            self.historial.pop(0)
        
        return respuesta

# Crear asistente con historial
asistente = RAGConHistorial(vector_store)

print("💬 RAG CON HISTORIAL DE CONVERSACIÓN")
print("=" * 50)
print()

# Simular conversación
preguntas = [
    "¿Cuántos días de vacaciones tengo?",
    "¿Y si me caso?",  # Referencia implícita al contexto anterior
    "¿Cuándo puedo pedirlas?"  # Más contexto implícito
]

for pregunta in preguntas:
    print(f"👤 Usuario: {pregunta}")
    respuesta = asistente.preguntar(pregunta)
    print(f"🤖 Asistente: {respuesta}")
    print()

---
# 📋 RESUMEN Y CONCLUSIONES

## 🎓 Lo que has aprendido

### ✂️ Chunking
- Dividir documentos en fragmentos manejables
- Métodos: tamaño fijo, por secciones, por oraciones, recursivo
- Sweet spot: 200-500 tokens con overlap

### 🔢 Embeddings para RAG
- Convertir chunks a vectores para búsqueda semántica
- Modelos gratuitos: HuggingFace Sentence Transformers
- Almacenar en vector store (simples o avanzados)

### 🔍 Retrieval
- Búsqueda por similitud coseno
- Umbrales y re-ranking
- Filtros por metadatos

### 🤖 Generación
- Componer prompt con contexto recuperado
- Instruir al modelo a usar SOLO el contexto
- Manejar historial de conversación

## 🏗️ Arquitectura típica de RAG en producción

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG EN PRODUCCIÓN                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  📄 Documentos → [Chunking] → [Embedding] → 🗄️ Vector DB   │
│                                              (Pinecone,     │
│                                               ChromaDB,     │
│                                               FAISS)        │
│                                                             │
│  👤 Query → [Embedding] → [Búsqueda] → [Chunks relevantes] │
│                                              ↓              │
│                                    [Prompt + Contexto]      │
│                                              ↓              │
│                                         🤖 LLM              │
│                                              ↓              │
│                                        📤 Respuesta         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 📚 Siguientes pasos

Para llevar RAG a producción, explora:
- **LangChain** / **LlamaIndex**: Frameworks completos de RAG
- **ChromaDB** / **Pinecone** / **Weaviate**: Vector databases escalables
- **Evaluación**: Métricas como faithfulness, relevance, coherence
- **Hybrid search**: Combinar embeddings con BM25

---

✨ **¡Felicidades!** Has completado el Notebook 4. Ahora entiendes cómo construir sistemas RAG completos.